Simple notebook to the repo that loads the GCM data you’ve catalogued alongside ERA5 daily data? it should load individual points and also a small region (let’s do South Africa)

In [ ]:
import duckdb
import geopandas as gpd
import icechunk
import rasterix
import xarray as xr
from rasterix.rasterize.rasterio import geometry_clip, geometry_mask

from srm import catalog
# import matplotlib.style as mplstyle
# mplstyle.use('fast')

In [ ]:
catalog

# Load ERA5 and CESM-WACCM

In [ ]:
def fix_coords(ds: xr.Dataset):
    ds.coords["lon"] = (ds.coords["lon"] + 180) % 360 - 180
    ds = ds.sortby(waccm.lon)
    return ds


waccm_cat = catalog.get("CESM2-WACCM-SSP245-icechunk")

waccm_storage = icechunk.s3_storage(
    bucket=waccm_cat.bucket,
    prefix=waccm_cat.prefix,
    from_env=True,
)
waccm_repo = icechunk.Repository.open(waccm_storage)
waccm_session = waccm_repo.readonly_session("main")
waccm = xr.open_zarr(waccm_session.store, consolidated=False)
waccm = fix_coords(waccm)
# waccm = rasterix.assign_index(waccm, x_dim='lon',y_dim='lat')
waccm = waccm.proj.assign_crs(spatial_ref="epsg:4326")

In [ ]:
waccm

In [ ]:
era5_cat = catalog.get("ERA5")
era5_storage = icechunk.s3_storage(
    bucket=era5_cat.bucket,
    prefix=era5_cat.prefix,
    from_env=True,
)
era5_repo = icechunk.Repository.open(era5_storage)
era5_session = era5_repo.readonly_session("main")
era5 = xr.open_zarr(era5_session.store).pipe(rasterix.assign_index)
era5 = era5.proj.assign_crs(spatial_ref="epsg:4326")
era5

## Load South Africa Geometry

In [ ]:
df = duckdb.sql(
    """install httpfs; load httpfs; install spatial; load spatial; SELECT name, ST_AsText(geom) as geometry FROM ST_Read('https://carbonplan-data.s3.us-west-2.amazonaws.com/countries-50m.json') WHERE NAME = 'South Africa'"""
).df()
df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"])
south_africa_geom = gpd.GeoDataFrame(df, geometry="geometry")
south_africa_geom

### Mask South-Africa

In [ ]:
geometry_mask(
    era5, south_africa_geom[["geometry"]], xdim="longitude", ydim="latitude"
).plot()

## Clip South-Africa Mask

In [ ]:
era5_south_africa = geometry_clip(
    era5, south_africa_geom[["geometry"]], xdim="longitude", ydim="latitude"
)
era5_south_africa.mean_total_precipitation_rate.isel(time=-1).plot(robust=True)

## Create a time-series for Cape Town

In [ ]:
cities = gpd.read_file(
    "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_populated_places_simple.zip",
    columns=["name", "sov0name", "geometry"],
)
sa_cities = cities[cities["sov0name"] == "South Africa"]
cape_town_x, cape_town_y = (
    float(sa_cities[sa_cities["name"] == "Cape Town"].geometry.x),
    float(sa_cities[sa_cities["name"] == "Cape Town"].geometry.y),
)

print(sa_cities[["name", "geometry"]])

In [ ]:
cape_town_time_series = era5_south_africa.sel(
    latitude=cape_town_y, longitude=cape_town_x, method="nearest"
)

In [ ]:
cape_town_time_series["2m_temperature"].drop_vars("spatial_ref").hvplot()

- 